In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('D:\GITHUB\LinkedIn Hiring Posts - Feb 2026(n8n Automation)\LinkedIn Hiring Posts - Feb 2026(n8n Automation)\ecommerce_messy_data.csv')

In [3]:
df.head()

,order_id,customer_name,email,phone,street_address,city,state,zip_code,product_name,category,subcategory,unit_price,quantity,discount_%,total_amount,shipping_cost,order_date,delivery_date,payment_method,order_status,customer_rating,review_count
0,306618,Patricia Brown,patricia.brown@gmail.com,+12462613434,8177 Cedar Dr,PHILADELPHIA,PA,55673,PRO SHOES,clothing,Shoes,509.28,6.0,21.2,2407.88,4.36,10/26/2022,11/17/2022,Debit Card,Cancelled,1.9,485
1,204530,James Smith,james.smith@gmail.com,5209519670,4308 Main Blvd,Phoenix,AZ,86569-9398,Car Parts Plus,AUTOMOTIVE,Car Parts,534.40,7.0,24.8,2813.08,0.94,NaN,2023-06-27,Credit Card,Shipped,4.7,2559
2,ord618933,MICHAEL BROWN,NaN,NaN,4857 Elm Ln,San Antonio,TX,26967-3096,ultra skincare x,Beauty,Skincare,153.47,4.0,7.2,569.68,2.98,NaN,-,debit card,Processing,4.2,1398
3,571689,Patricia Smith,patricia@yahoo.com,(499) 781-7874,8724 Elm Ave,New York,NY,99267,Basic Water Sports,Sports,Water Sports,850.18,1.0,NaN,850.18,14.45,2022-09-20,2022-10-14,Google Pay,Cancelled,1.2,-
4,ord197194,Linda Garcia,linda.garcia@gmail.com,(861) 155-4625,9545 Main Ave,Dallas,TX,NaN,Basic International Max,Food,International,382.78,5.0,-,1913.90,0.87,2024-12-25,01/15/2025,Bank Transfer,Returned,3.5,846


In [5]:
df.columns

Index(['order_id', 'customer_name', 'email', 'phone', 'street_address', 'city',
       'state', 'zip_code', 'product_name', 'category', 'subcategory',
       'unit_price', 'quantity', 'discount_%', 'total_amount', 'shipping_cost',
       'order_date', 'delivery_date', 'payment_method', 'order_status',
       'customer_rating', 'review_count'],
      dtype='object')

### ID column

In [4]:
## Step 1 — Explore

# Check how many nulls exist
print("Number of null values in each column:")
print(df.isnull().sum())

Number of null values in each column:
order_id             30
customer_name        30
email              1393
phone              1507
street_address      723
city                282
state               358
zip_code           1337
product_name         35
category             30
subcategory         588
unit_price           30
quantity             30
discount_%          902
total_amount         30
shipping_cost       700
order_date          995
delivery_date       927
payment_method      348
order_status        350
customer_rating    1642
review_count       1260
dtype: int64


In [5]:
# Check how many duplicates exist
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 29


In [6]:
## Step 2 — Standardize

# Remove all prefixes and non-numeric characters (assumption documented — client confirmed prefixes carry no meaning)
df['order_id'] = df['order_id'].str.replace(r'[^\d]', '', regex=True)
# Keep the column as string type
df['order_id'] = df['order_id'].astype("string") # Using pandas string type to preserve NULL values

In [7]:
## Step 3 — Handle Nulls

# Drop rows where order_id is null or blank after standardization
original_count = len(df)
df = df[df['order_id'].notna() & (df['order_id'] != '')]
new_count = len(df)
print(f"Rows dropped: {original_count - new_count}")

Rows dropped: 30


In [8]:
## Step 4 — Handle Duplicates

# Identical rows → keep first occurrence, drop the rest
df.drop_duplicates(inplace=True)
# Same ID but different row data → flag them, do not delete, escalate to client
duplicate_ids = df[df.duplicated(subset=['order_id'], keep=False)]['order_id'].unique()
print("Duplicate order IDs:", duplicate_ids)

Duplicate order IDs: <StringArray>
['367762', '903603', '127560', '193108', '138760', '790306', '627761',
 '936923', '335513', '520538',
 ...
 '612349', '555910', '768124', '231185', '812661', '567778', '182919',
 '464361', '974151', '786660']
Length: 375, dtype: string


In [9]:
## Step 5 — Validate

#Confirm zero nulls remain
print("Number of null values in each column after cleaning:")
print(df.isnull().sum())
# #Confirm zero non-numeric characters remain
print("Unique characters in order_id after cleaning:", df['order_id'].apply(lambda x: set(x)).explode().unique())
#Compare duplicate count before and after
print("Number of duplicate rows after cleaning:", df.duplicated().sum())
#Compare total row count before and after — explain any significant drop
print(f"Total rows before cleaning: {original_count}, after cleaning: {len(df)}")

Number of null values in each column after cleaning:
order_id              0
customer_name         0
email              1363
phone              1477
street_address      693
city                252
state               328
zip_code           1307
product_name          5
category              0
subcategory         558
unit_price            0
quantity              0
discount_%          872
total_amount          0
shipping_cost       670
order_date          965
delivery_date       897
payment_method      318
order_status        320
customer_rating    1612
review_count       1230
dtype: int64
Unique characters in order_id after cleaning: ['0' '6' '1' '8' '3' '5' '2' '4' '9' '7']
Number of duplicate rows after cleaning: 0
Total rows before cleaning: 10000, after cleaning: 9970


### Order Date

###

In [10]:
# Find out all the null characters in the order_date column
print("Number of null values in the dataframe:  ",df["order_date"].isnull().sum())

Number of null values in the dataframe:   965


In [11]:
# Find out all the different formats in the order_date column
print("Unique formats in order_date column:")
df["length"] = df["order_date"].str.len()
print(df.groupby("length").size())

Unique formats in order_date column:
length
1.0       49
10.0    7960
12.0      89
13.0     153
14.0     155
15.0      88
16.0     177
17.0     243
18.0      91
dtype: int64


In [15]:
# converted = pd.to_datetime(df["order_date"], errors="coerce")
# # df[converted.isna() & df["order_date"].notna()]["order_date"].value_counts()
# converted.min(), converted.max()

In [12]:
df['order_date'].unique()

array(['10/26/2022', nan, '2022-09-20', ..., '2022-11-03',
       'November 18, 2022', '09-11-2022'], dtype=object)

In [13]:
from dateutil import parser
import pandas as pd

def parse_date_safe(val):
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'None'):
        return pd.NaT
    try:
        # dayfirst=False ensures MM/DD/YYYY takes priority over DD/MM/YYYY
        # where ambiguous (e.g. 10/06/2022 → Oct 6, not June 10)
        return parser.parse(str(val).strip(), dayfirst=False)
    except (ValueError, OverflowError):
        return pd.NaT

df['order_date_cleaned'] = (
    df['order_date']
    .astype(str)
    .str.strip()
    .str.replace(r'\xa0', '', regex=True)         # Removes non-breaking spaces — invisible characters that sometimes sneak in from websites or Excel
    .str.replace(r'^-$', '', regex=True)          # If a cell contains just a dash ("-"), replace it with empty string. A lone dash usually means "no data"
    .map(parse_date_safe)                         # Runs our safe function on every single value to convert text → date
    .dt.normalize()                                # strips time component
    .dt.strftime('%Y-%m-%d')                       # final yyyy-mm-dd string
)

In [14]:
df[['order_date', 'order_date_cleaned']].head(50)

,order_date,order_date_cleaned
0,10/26/2022,2022-10-26
1,NaN,NaN
2,NaN,NaN
3,2022-09-20,2022-09-20
4,2024-12-25,2024-12-25
5,04/16/2024,2024-04-16
6,2024-12-25,2024-12-25
7,NaN,NaN
8,2024-04-10,2024-04-10
9,2022-03-04,2022-03-04


In [18]:
df[df['order_date_cleaned'].isna()]['order_date'].unique()

array([nan, '13/32/2023', '-'], dtype=object)

In [77]:
df[df["order_id"].str.contains(r'[^\d]', na=False)].sort_values(by="order_id")

,order_id,customer_name,email,phone,street_address,city,state,zip_code,product_name,category,subcategory,unit_price,quantity,discount_%,total_amount,shipping_cost,order_date,delivery_date,payment_method,order_status,customer_rating,review_count
5226,#100111,JAMES MOORE,james.moore@gmail.com,(331) 241-9619,3678 Main Blvd,Phoenix,AZ,51781-9819,Smartphones Plus,Electronics,Smartphones,286.64,6.0,5.2,1630.41,19.25,NaN,09/11/2022,Credit Card,RETURNED,4.5,2449
7156,#100452,PATRICIA MILLER,INVALID_EMAIL,NaN,77 Maple Blvd,Los Angeles,CA,NaN,OUTDOOR FURNITURE X,Garden,Outdoor Furniture,802.09,8.0,18.4,5236.04,8.26,2024-12-18,12/15/2024,PAYPAL,Returned,3.6,3151
3855,#100801,James Moore,james.moore@gmail.com,305-057-7489,333 Cedar Blvd,Chicago,IL,NaN,TEXTBOOKS PLUS,Books,Textbooks,63.25,10.0,10.5,566.09,19.56,2024-06-03,2024-07-02,Google Pay,Shipped,3.2,4784
3541,#101127,Mary Davis,mary@yahoo.com,(259) 065-6945,NaN,Dallas,TX,26811,PRO FICTION,Books,Fiction,930.58,7.0,27.5,4722.69,9.59,NaN,01/10/2025,PayPal,NaN,4.9,672
9734,#101368,John Wilson,john.wilson@gmail.com,NaN,3229 Oak Dr,new york,NY,NaN,Ultra Tools,Garden,Tools,440.16,9.0,13.8,3414.76,14.32,"October 22, 2023",NaN,NaN,Processing,4.0,2145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9772,-998875,Linda Smith,linda@yahoo.com,(711) 089-6709,8645 Cedar Ln,SAN ANTONIO,TX,76721-1451,Ultra Educational,TOYS,Educational,118.50,6.0,1.0,703.89,7.01,2022-01-21,2022-02-09,PayPal,PROCESSING,NaN,1011
6186,-999187,Robert Wilson,robert@yahoo.com,7090371425,5400 Cedar St,Chicago,IL,55713,jackets plus,Clothing,Jackets,462.93,1.0,22.2,331.76,10.89,2023-07-09,2023-07-26,PayPal,returned,4.2,4381
9769,-999507,Jennifer Smith,INVALID_EMAIL,NaN,2354 Main Blvd,SAN ANTONIO,TX,68024,Premium Accessories Plus,Automotive,Accessories,345.85,8.0,27.7,2000.40,27.81,"May 17, 2022",NaN,Debit Card,Shipped,3.9,NaN
3465,-999867,William Davis,william.davis@gmail.com,NaN,9144 Maple Dr,San Antonio,TX,66559,Pro Supplements X,Food,Supplements,635.93,3.0,NaN,1907.79,25.11,2022-10-05,10/30/2022,Debit Card,cancelled,3.0,NaN


In [51]:

#  delete all the customers with NaN values in all the columns
df.dropna(how="all", inplace=True)

In [53]:
# Find out the order with same order_id
df[df["order_id"].duplicated(keep=False)].sort_values(by="order_id")

# Find out the order with same email
df[df["email"].duplicated(keep=False)].sort_values(by=["email", "customer_name"])


,order_id,customer_name,email,phone,street_address,city,state,zip_code,product_name,category,subcategory,unit_price,quantity,discount_%,total_amount,shipping_cost,order_date,delivery_date,payment_method,order_status,customer_rating,review_count
1230,#101784,BARBARA SMITH,barbara.smith@gmail.com,992-378-8451,6867 Maple Blvd,chicago,IL,71292,Premium Car Parts Max,automotive,Car Parts,177.94,5.0,27.7,643.25,0.24,"June 14, 2022",06/15/2022,Apple Pay,Returned,1.5,3950
4330,#101784,Jennifer Miller,jennifer@yahoo.com,+18779967921,7002 Elm St,NEW YORK,NY,84976,Pro Cookware Max,Home & Kitchen,Cookware,0.00,7.0,26.6,0.00,13.82,2022-05-06,05/24/2022,Apple Pay,Pending,1.7,3863
7970,#101906,JOHN MILLER,INVALID_EMAIL,202-863-6472,6683 Elm Ln,san jose,CA,19926-5011,Ultra Dolls,Toys,Dolls,386.30,2.0,15.5,652.85,26.04,13/32/2023,03/12/2023,Debit Card,PROCESSING,2.3,3450
6452,#101906,barbara brown,barbara@,905-585-4033,4582 Cedar St,Los Angeles,CA,85765-4260,Premium Snacks Max,food,Snacks,416.34,2.0,3.9,800.21,3.32,"April 27, 2023",2023-05-22,apple pay,PROCESSING,3.4,2381
8861,#107723,Robert Brown,robert@yahoo.com,NaN,1214 Oak Ln,SAN ANTONIO,TX,3852,Basic Textbooks X,Books,Textbooks,934.00,9.0,29.5,5926.23,2.21,2024-10-13,2024-10-16,Credit Card,Returned,2.0,1132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8646,ord964819,BARBARA MOORE,barbara@yahoo.com,959-343-9331,6725 Main St,Los Angeles,CA,84020-1652,Pro Outdoor Gear Max,Sports,Outdoor Gear,157.22,8.0,28.2,665.45,14.01,"October 23, 2023",11/06/2023,apple pay,delivered,1.2,NaN
2423,ord990232,WILLIAM SMITH,INVALID_EMAIL,(203) 240-5282,9340 Elm Ave,SAN DIEGO,CA,42382,Pro Car Parts Plus,Automotive,Car Parts,682.94,7.0,29.6,3365.53,27.25,10/29/2023,11/08/2023,paypal,Processing,2.8,1634
8936,ord990232,John Smith,john.smith@gmail.com,(251) 501-7819,1103 Cedar St,Los Angeles,CA,23166-1756,Basic Organic,Food,Organic,433.38,7.0,16.4,2536.14,5.03,09/11/2022,10/08/2022,Debit Card,Cancelled,-,NaN
5640,ord998278,Robert Smith,NaN,(723) 973-9169,NaN,Phoenix,AZ,23814,tablets,Electronics,Tablets,596.41,9.0,29.2,3800.32,5.39,2024-03-15,03/13/2024,paypal,Cancelled,2.2,602


In [66]:
df[df.duplicated(subset=['email', "order_id"], keep=False)].sort_values(by=["email", "order_id"])

,order_id,customer_name,email,phone,street_address,city,state,zip_code,product_name,category,subcategory,unit_price,quantity,discount_%,total_amount,shipping_cost,order_date,delivery_date,payment_method,order_status,customer_rating,review_count
1060,160941,John Davis,INVALID_EMAIL,1234567890,868 Oak Ln,Philadelphia,PA,39726-8255,Men's Shirts Max,clothing,Men's Shirts,938.46,6.0,24.7,4239.96,12.01,2022-04-01,2022-04-27,Gift Card,Refunded,3.3,2648
2264,160941,John Davis,INVALID_EMAIL,(932) 339-6679,7942 Cedar Dr,san diego,CA,NaN,basic furniture x,Home & Kitchen,Furniture,876.96,7.0,22.8,4739.09,14.7,2024-11-27,2024-11-28,Google Pay,Delivered,4.8,2705
1912,ORD-736189,Linda Brown,INVALID_EMAIL,(720) 387-1534,1899 Maple Dr,Chicago,IL,33765,Headphones Max,Electronics,Headphones,0.00,7.0,10.0,0.00,26.09,NaN,2023-05-04,Credit Card,Delivered,1.8,2837
6998,ORD-736189,James Moore,INVALID_EMAIL,(685) 131-6437,2603 Main Ave,San Diego,CA,93088-5489,ULTRA CHILDREN'S PLUS,Books,Children's,391.89,4.0,10.6,1401.40,15.87,2024-02-24,2024-03-24,Credit Card,Processing,NaN,NaN
7391,ord310214,Mary Davis,INVALID_EMAIL,+19502569345,8431 Oak Blvd,Dallas,TX,66853,basic tools x,Beauty,Tools,669.59,5.0,13.2,2906.02,27.22,2022-08-17,2022-08-13,Bank Transfer,refunded,NaN,3253
8649,ord310214,Patricia Smith,INVALID_EMAIL,(498) 566-2024,75 Maple Blvd,San Jose,CA,NaN,Pro Cycling Plus,SPORTS,Cycling,433.15,8.0,16.1,2907.30,1.48,13/32/2023,2024-08-27,Gift Card,REFUNDED,1.5,1727
1706,ord949776,Linda Smith,INVALID_EMAIL,NaN,3479 Cedar Dr,san diego,CA,NaN,ultra clothing max,Clothing,NaN,494.37,6.0,10.7,2648.83,9.7,2023-03-29,2023-04-27,Gift Card,Cancelled,3.4,NaN
3391,ord949776,Patricia Davis,INVALID_EMAIL,8945283603,3010 Oak Ave,San Antonio,TX,NaN,pro storage plus,HOME & KITCHEN,Storage,818.61,8.0,1.1,6476.84,8.59,"August 14, 2024",09/05/2024,Gift Card,Returned,2.8,4840
1658,ORD-222812,Barbara Moore,barbara@yahoo.com,491-923-3578,9176 Main Dr,San Diego,CA,NaN,ULTRA LEGO MAX,Toys,LEGO,811.83,7.0,24.5,4290.52,12.9,"May 16, 2023",2023-06-15,Bank Transfer,PROCESSING,NaN,147
9397,ORD-222812,Barbara Garcia,barbara@yahoo.com,(978) 869-7076,8346 Cedar St,Phoenix,AZ,99906,Ultra Fiction X,Books,Fiction,272.95,9.0,NaN,2456.55,0.0,14-07-2022,08/01/2022,Bank Transfer,Refunded,-,828


In [55]:
# Let us check for any blank values in the order_id column
print("Blank order_id values:", df["order_id"].isna().sum())
#Let us drop any blank values in the order_id column
df.dropna(subset=["order_id"], inplace=True)

#let us check for any non-numeric values in the order_id column
print("Non-numeric order_id values:", df["order_id"].str.contains(r'[^\d]').sum())
# # Let is replace any non-numeric values with blank
df["order_id"] = df["order_id"].str.replace(r'[^\d]', '', regex=True)
 
# Let us convert the order_id column to string type
df["order_id"] = df["order_id"].astype(str)

# Let us check for any duplicates in the order_id column
print("Number of duplicate order_id values:", df["order_id"].duplicated().sum())
# let us drop them if there are any
df.drop_duplicates(subset=["order_id"], inplace=True)

# print("Distinct order_id values:", df["order_id"].nunique())


# print("*" * 50)
# print("Non-numeric order_id values:", df["order_id"].str.contains(r'[^\d]').sum())
# print("Blank order_id values:", df["order_id"].isna().sum())
# print("Number of duplicate order_id values:", df["order_id"].duplicated().sum())
# print("Distinct order_id values:", df["order_id"].nunique())



Blank order_id values: 30
Non-numeric order_id values: 7509


### Customer Name

In [11]:
# make all characters lower‑case and then capitalise the first letter
df['customer_name'] = df['customer_name'].str.title()
# remove leading and trailing spaces
df['customer_name'] = df['customer_name'].str.strip()




### email_id

In [18]:
df[["customer_name", "email"]]

,customer_name,email
0,Patricia Brown,patricia.brown@gmail.com
1,James Smith,james.smith@gmail.com
2,Michael Brown,NaN
3,Patricia Smith,patricia@yahoo.com
4,Linda Garcia,linda.garcia@gmail.com
...,...,...
9995,Linda Williams,linda.williams@gmail.com
9996,Jennifer Williams,jennifer.williams@gmail.com
9997,Robert Johnson,NaN
9998,Robert Miller,robert@yahoo.com


In [15]:
print("Distinct email domains:", df["email"].str.split("@").str[1].value_counts())


Distinct email domains: email
gmail.com    4694
yahoo.com    1959
              930
Name: count, dtype: int64
